# Example 2: With Cell Metadata - Stroke Dataset

This notebook demonstrates compiling a modern GEO dataset where each sample has paired count and cell metadata files.

**Dataset**: GSE225948 (Stroke brain and blood samples)  
**Format**: Paired *_counts.csv.gz and *_metadata.csv.gz files per sample  
**Samples**: 3 samples (2 brain, 1 blood)

The pipeline compiles raw data into a QC-filtered, normalized h5ad. Downstream analysis (PCA, UMAP, clustering) is shown as an optional follow-up.

In [ ]:
import sys
import os
sys.path.append('../../')  # Add repo to path

from anndata_compiler import GEOAnndataCompiler
import pandas as pd

## 1. Examine the Data Structure

In [ ]:
# Look at the files
data_dir = '../data/with_cell_metadata_stroke'
print("Files in data directory:")
files = sorted(os.listdir(data_dir))
for f in files:
    print(f"  {f}")

print(f"\nFile pairs:")
counts_files = [f for f in files if '_counts.csv.gz' in f]
metadata_files = [f for f in files if '_metadata.csv.gz' in f]
print(f"  Count files: {len(counts_files)}")
print(f"  Metadata files: {len(metadata_files)}")

In [ ]:
# Look at sample-level metadata
metadata = pd.read_csv(f'{data_dir}/metadata.csv')
print(f"Sample-level metadata ({metadata.shape[0]} samples):")
print(metadata)

## 2. Configure the Compiler

The compiler will auto-detect the paired file format. QC filtering is applied automatically:

In [ ]:
config = {
    'raw_data_dir': data_dir,
    'metadata_file': f'{data_dir}/metadata.csv',
    'output_file': './stroke_compiled_example.h5ad',
    'sample_id_column': 'Sample_ID',
    
    # Processing parameters
    'max_cells_per_sample': 300,  # Small for demo; use None for full data
    'target_sum': 1e4,
    'n_top_genes': 2000,
    'delimiter': ',',
    
    # QC filtering
    'min_genes': 200,
    'min_cells': 3,
    'max_mito_pct': 20.0,
    
    # Auto-detection will find the paired files
    'data_format': 'auto',
    
    # Selective metadata inclusion
    'metadata_columns': ['Tissue', 'Condition', 'Sex']
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Run the Compilation Pipeline

In [ ]:
# Initialize compiler
compiler = GEOAnndataCompiler(config)

# Run pipeline (compiles, QC filters, normalizes, detects HVGs)
adata = compiler.run_full_pipeline()

print(f"\nFinal dataset: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Layers: {list(adata.layers.keys())}")
print(f"Samples: {adata.obs['sample_id'].unique()}")
print(f"Tissues: {adata.obs['Tissue'].unique()}")

## 4. Inspect the Compiled Data

In [ ]:
import scanpy as sc

print("AnnData structure:")
print(adata)
print(f"\nObservation columns: {list(adata.obs.columns)}")
print(f"Variable columns: {list(adata.var.columns)}")
print(f"Layers: {list(adata.layers.keys())}")
print(f"HVGs: {adata.var['highly_variable'].sum()}")

# Cell metadata columns from GEO files
cell_meta_cols = [col for col in adata.obs.columns if col not in ['sample_id', 'Tissue', 'Condition', 'Sex']]
print(f"\nCell metadata from GEO files: {cell_meta_cols}")

In [ ]:
# Sample and tissue composition
composition = adata.obs.groupby(['sample_id', 'Tissue']).size().unstack(fill_value=0)
print("Cells per sample and tissue:")
print(composition)

if 'cell_type' in adata.obs.columns:
    print(f"\nCell types found: {adata.obs['cell_type'].unique()}")
    print(adata.obs['cell_type'].value_counts())

In [ ]:
# QC visualization (post-filtering)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
if 'cell_type' in adata.obs.columns:
    print("Cell type distribution:")
    print(adata.obs['cell_type'].value_counts())

## 5. Compare Sample-level vs Cell-level Metadata

In [ ]:
print("Sample-level metadata (added to all cells in each sample):")
sample_meta = adata.obs[['sample_id', 'Tissue', 'Condition', 'Sex']].drop_duplicates()
print(sample_meta)

print("\nExample cell-level metadata (varies per cell):")
if len(cell_meta_cols) > 0:
    example_cols = ['sample_id'] + cell_meta_cols[:3]
    print(adata.obs[example_cols].head(10))
else:
    print("No cell-level metadata columns found in this example.")

## 6. Optional: Downstream Analysis Preview

The compiled h5ad is the output of the pipeline. Everything below is interactive analysis.
See [docs/downstream_analysis_guide.md](../../docs/downstream_analysis_guide.md) for full details.

In [ ]:
# PCA and scree plot
sc.tl.pca(adata, svd_solver='arpack', n_comps=30, use_highly_variable=True)
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)

# Neighbors, UMAP, and clustering
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata)

for res in [0.3, 0.5, 1.0]:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.umap(adata, color='leiden_0.5', ax=axes[0], show=False, frameon=False)
axes[0].set_title('Leiden (res=0.5)')
sc.pl.umap(adata, color='Tissue', ax=axes[1], show=False, frameon=False)
axes[1].set_title('Tissue Type')
sc.pl.umap(adata, color='sample_id', ax=axes[2], show=False, frameon=False)
axes[2].set_title('Sample ID')
plt.tight_layout()
plt.show()